In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import logging
import random
from copy import deepcopy
import re
from sklearn.metrics import mean_squared_error, root_mean_squared_error

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import numpy as np
from dotenv import load_dotenv
from natsort import natsorted
from halcyon.Simulators.SimulationResult import SimulationResult
from halcyon.results.manipulate_results_df import (create_tidy_df_from_mlflow_runs, 
                                                   split_df_into_single_and_double_ko)

from halcyon.results import DfColumnNames

seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 

load_dotenv() 

logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)
os.environ['MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR'] = 'False'



In [74]:
def parse_condition(condition: str) -> tuple[tuple[str, ...], str]:
    """Split a condition string into knocked-out genes and environment."""
    ko_part, _, env_part = condition.partition(". Env: ")
    ko_genes = tuple(re.findall(r"Node_\d+", ko_part.replace("KO: ", "")))
    return ko_genes, env_part or "None"


def compute_gene_level_double_ko_epistasis(df: pd.DataFrame) -> pd.DataFrame:
    """For each gene, compare the double-KO trajectory to both single-KO trajectories and keep the strongest match."""
    wide_df = df.pivot_table(
        index=[DfColumnNames.TIME_COL],
        columns=[DfColumnNames.GENE_COL, DfColumnNames.CONDITION_COL],
        values=DfColumnNames.GROUND_TRUTH_COL,
        aggfunc="first",
    )

    rows = []
    conditions = list(df[DfColumnNames.CONDITION_COL].unique())
    genes = sorted(df[DfColumnNames.GENE_COL].unique())

    for condition in sorted(conditions):
        ko_genes, env_name = parse_condition(condition)

        if len(ko_genes) != 2:
            continue

        matching_single_conditions = []
        for single_gene in ko_genes:
            matching_single_condition = next(
                (
                    candidate
                    for candidate in conditions
                    if parse_condition(candidate)[0] == (single_gene,)
                    and parse_condition(candidate)[1] == env_name
                ),
                None,
            )
            if matching_single_condition is not None:
                matching_single_conditions.append(matching_single_condition)

        for gene_name in genes:
            if (gene_name, condition) not in wide_df.columns:
                continue

            single_mses = []
            for single_condition in matching_single_conditions:
                if (gene_name, single_condition) not in wide_df.columns:
                    continue

                double_series = wide_df[(gene_name, condition)]
                single_series = wide_df[(gene_name, single_condition)]

                comparison = pd.concat(
                    [
                        double_series.rename("double_ko"),
                        single_series.rename("single_ko"),
                    ],
                    axis=1,
                ).dropna()

                mse = root_mean_squared_error(comparison["double_ko"], comparison["single_ko"])
                single_mses.append((mse, single_condition))

            lowest_mse, best_single_condition = min(
                single_mses,
                key=lambda item: item[0],
            )
            rows.append(
                {
                    'condition': condition,
                    "double_ko": ", ".join(ko_genes),
                    "gene": gene_name,
                    "best_single_ko_condition": best_single_condition,
                    "single-to-double-ko-divergence": lowest_mse
                }
            )

    return pd.DataFrame(rows).sort_values("single-to-double-ko-divergence", ascending=False)


Just have to get the simulation run of the 50 gene network I suppose

In [75]:
records = []
run_ids = ['6a8078c2c7764850b1fb8b3c33553a1b', 'f02bf732b2ba4817b254bd5174f1549b', 'f862625665764f1383bf88e5615594dc']
for run_id in run_ids:
    ml_flow_download_path = mlflow.artifacts.download_artifacts(
        run_id=run_id, artifact_path="dataset"
    )
    sim_output = SimulationResult.load_from_pkl(
        Path(ml_flow_download_path) / "simulation_result.pkl"
    )
    sim_output.noiseless_for_testing = True
    long_form_simulation = sim_output.to_long_form_df()
    long_form_simulation[DfColumnNames.RUN_ID_COL] = run_id
    epistasis_df = compute_gene_level_double_ko_epistasis(long_form_simulation)
    epistasis_df['nr_genes'] = mlflow.get_run(run_id).data.params['nr_genes']
    records.append(epistasis_df)
full_df = pd.concat(records, ignore_index=True)

full_df.to_csv('epistasis_levels.csv')

2026-07-01 12:10:59,829 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-07-01 12:10:59,856 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-07-01 12:10:59,861 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-07-01 12:10:59,872 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-07-01 12:10:59,900 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-07-01 12:10:59,917 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-07-01 12:10:59,932 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-07-01 12:10:59,937 - WARNING - Connection p

10{"stdout":"[{\"variableName\": \"ID_TO_MEANING\", \"type\": \"dictionary\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"builtins.dict\"}, {\"variableName\": \"NULL\", \"type\": \"unknown\", \"supportedEngines\": [\"pandas\"], \"isLocalVariable\": true, \"rawType\": \"_pydevd_bundle.pydevd_constants.Null\"}]\n","stderr":"","mime":[]}


In [76]:
full_df['nr_genes'].value_counts()

nr_genes
100    10000
50      2500
20       400
Name: count, dtype: int64

In [ ]:
sns.displot(epistasis_df, x='single-to-double-ko-divergence')

In [ ]:
a = long_form_simulation[((long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_7, Node_23. Env: None') | (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_23. Env: None')| (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_7. Env: None')) & (long_form_simulation[DfColumnNames.GENE_COL] == 'Node_45') ]
a
sns.relplot(data=a, x='time', y='ground_truth', col='condition', kind='line')

In [ ]:
a = long_form_simulation[((long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_13, Node_24. Env: None') | (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_24. Env: None')| (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_13. Env: None')) & (long_form_simulation[DfColumnNames.GENE_COL] == 'Node_24') ]
a
sns.relplot(data=a, x='time', y='ground_truth', col='condition', kind='line')

In [ ]:
a = long_form_simulation[((long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_4, Node_24. Env: None') | (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_4. Env: None') | (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_24. Env: None')) & (long_form_simulation[DfColumnNames.GENE_COL] == 'Node_15') ]
a
sns.relplot(data=a, x='time', y='ground_truth', col='condition', kind='line')

In [ ]:
a = long_form_simulation[((long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_6, Node_23. Env: None') | (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_23. Env: None') | (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_6. Env: None')) & (long_form_simulation[DfColumnNames.GENE_COL] == 'Node_25') ]
a
sns.relplot(data=a, x='time', y='ground_truth', col='condition', kind='line')

In [ ]:
a = long_form_simulation[((long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_6, Node_23. Env: None') | (long_form_simulation[DfColumnNames.CONDITION_COL] == 'KO: Node_6. Env: None')) & (long_form_simulation[DfColumnNames.GENE_COL] == 'Node_25') ]
a
sns.relplot(data=a, x='time', y='ground_truth', col='condition', kind='line')

In [ ]:
epistasis_df.to_csv('epistasis_levels.csv')